# Run and evaluate the Energy Advisor
Run the setup notebooks first. Use the notebook toolbar's **Select Kernel → Python (EcoHome)**, restart the kernel, and run cells from the top. The questions cover scheduling, thermostat guidance, history-based advice, savings, and solar-aware pool-pump planning.

In [3]:
import json
import os
import sys
from pathlib import Path
from IPython.display import Markdown, display
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

expected_environment = Path.cwd() / ".venv"
if expected_environment.is_dir() and Path(sys.prefix).resolve() != expected_environment.resolve():
    expected_python = expected_environment / "Scripts" / "python.exe"
    raise RuntimeError(
        "Wrong notebook kernel. The running interpreter is "
        f"{sys.executable!r}; expected {str(expected_python)!r}. "
        "In the notebook toolbar, open Select Kernel, choose Python (EcoHome), "
        "then choose Restart Kernel and run all cells."
    )
load_dotenv(dotenv_path=Path.cwd() / ".env")
from agent import Agent, build_energy_advisor

ECOHOME_SYSTEM_PROMPT = """You are EcoHome's household energy advisor. Answer directly, concisely, and in actionable terms using only the provided question and graph evidence.

Ground numeric claims in supplied energy data or clearly identified user assumptions. Bundled tariff, solar, and weather are synthetic sample forecasts, not live feeds. Label cost, savings, solar-use, and carbon figures as estimates; name the baseline and time horizon. Use 0.4 kg CO2e/kWh only as an illustrative solar-offset assumption.

Cite retrieved knowledge-base filenames when using their guidance. Never invent citations, rates, device history, or forecast values; state when evidence is missing. Prioritize comfort, health, food safety, EV departure charge, pool circulation, equipment limits, and battery reserves over savings. Thermostat values are optional starting points, not medical advice. Do not defer required loads beyond user deadlines. Calculate ROI only from supplied investment, annual savings, and incentives; state that simple payback excludes financing, maintenance, degradation, and taxes. Never promise savings.

Example questions and response expectations:
- 'When should I charge my electric car tomorrow?' Use EV schedule, solar, and price evidence; give a schedule, savings baseline, and citations.
- 'How much can I save by running my dishwasher off-peak?' State window and estimated savings versus an 18:00 same-duration baseline.
- 'What should I set my thermostat to during a price spike?' Give a comfort-qualified optional setpoint using forecast context.
- 'How can I reduce use based on history?' Identify the highest measured load and provide safe, history-grounded actions.
- 'What is the ROI for a battery?' Calculate only from user-provided financial inputs and explain limitations.
- 'When should my pool pump run this week?' Use the seven-day schedule and cite forecast-based recommendations."""

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in the notebook kernel environment before running the LLM-backed evaluation.")
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

forecast_tools = ["get_weather_forecast", "get_electricity_prices", "get_solar_generation", "retrieve_energy_tips"]
scenarios = [
    {"id": "ev-solar-tomorrow", "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?", "expected_tools": ["optimize_device_schedule", *forecast_tools], "expected_response": ["EV Charger", "estimated savings", "forecast solar", "Knowledge base:"]},
    {"id": "dishwasher-off-peak", "question": "How much can I save by running my dishwasher during off-peak hours?", "expected_tools": ["optimize_device_schedule", *forecast_tools], "expected_response": ["Dishwasher", "estimated savings", "18:00 start", "Knowledge base:"]},
    {"id": "pool-pump-weekly", "question": "What's the best time to run my pool pump this week based on the weather forecast?", "expected_tools": ["optimize_device_schedule_this_week", *forecast_tools], "expected_response": ["Pool Pump", "estimated savings", "Forecast:", "Knowledge base:"]},
    {"id": "thermostat-price-spike", "question": "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?", "expected_tools": forecast_tools, "expected_response": ["setpoint", "comfort and health needs", "forecast averages", "Knowledge base:"]},
    {"id": "usage-history-actions", "question": "Suggest three ways I can reduce energy use based on my usage history.", "expected_tools": [*forecast_tools, "summarize_usage_history"], "expected_response": ["highest measured load", "Three practical actions", "Knowledge base:"]},
    {"id": "washer-schedule", "question": "Schedule my washer for the lowest-cost time tomorrow.", "expected_tools": ["optimize_device_schedule", *forecast_tools], "expected_response": ["Washer", "estimated energy cost", "estimated savings", "Knowledge base:"]},
    {"id": "hvac-weather-guidance", "question": "What thermostat setting should I use tomorrow afternoon to save energy without sacrificing comfort?", "expected_tools": forecast_tools, "expected_response": ["setpoint", "comfort and health needs", "forecast averages", "Knowledge base:"]},
    {"id": "battery-roi", "question": "Estimate ROI and payback for a home battery.", "inputs": {"investment_cost_usd": 1200, "annual_savings_usd": 400, "incentives_usd": 200}, "expected_tools": [*forecast_tools, "summarize_usage_history", "calculate_roi"], "expected_response": ["net cost $1000.00", "simple payback 2.5 years", "excludes financing", "Knowledge base:"]},
    {"id": "history-based-reduction", "question": "Based on my recent energy usage history, give me practical steps to reduce consumption.", "expected_tools": [*forecast_tools, "summarize_usage_history"], "expected_response": ["highest measured load", "Three practical actions", "flexible loads", "Knowledge base:"]},
    {"id": "ev-weekly-plan", "question": "Find the best day and time to charge my EV this week, considering cost and solar.", "expected_tools": ["optimize_device_schedule_this_week", *forecast_tools], "expected_response": ["EV Charger", "estimated savings", "forecast solar", "Knowledge base:"]},
]
advisor = build_energy_advisor(model=model, instructions=ECOHOME_SYSTEM_PROMPT)
response_evaluator = Agent(
    """Evaluate one EcoHome response against the question, reference criteria, and actual graph evidence. Be strict and evidence-based. Return only a JSON object with integer scores from 0 to 100 for accuracy, relevance, completeness, and usefulness; an explanations object mapping each score name to a concise justification; and strengths and gaps string arrays. Accuracy means no unsupported or contradictory material claims. Relevance means it addresses the question. Completeness means it covers the expected_response reference criteria and important constraints. Usefulness means actions are clear, practical, appropriately caveated, and grounded. Do not reward mere keyword presence.""",
    model,
)
recommendations_agent = Agent(
    """You are an evaluation analyst. Based only on the supplied scored scenario evidence, give three to five prioritized, specific improvement recommendations. Cite scenario IDs and metric/tool evidence for each recommendation. Do not claim defects unsupported by evidence; if scores are strong, recommend targeted edge-case coverage. Return concise Markdown bullets.""",
    model,
)

In [4]:
def evaluate_response_quality(scenario, response, context):
    evaluation = response_evaluator.invoke(
        "Evaluate this response against the reference criteria. Return only valid JSON.",
        {"question": scenario["question"], "reference_criteria": scenario["expected_response"], "response": response, "evidence": context},
    )
    assessment = json.loads(evaluation)
    dimensions = ("accuracy", "relevance", "completeness", "usefulness")
    for dimension in dimensions:
        score = assessment.get(dimension)
        if not isinstance(score, (int, float)) or not 0 <= score <= 100:
            raise ValueError(f"Evaluator returned an invalid {dimension} score: {score!r}")
        if not isinstance(assessment.get("explanations", {}).get(dimension), str):
            raise ValueError(f"Evaluator omitted the {dimension} explanation.")
    for key in ("strengths", "gaps"):
        if not isinstance(assessment.get(key), list) or not all(isinstance(item, str) for item in assessment[key]):
            raise ValueError(f"Evaluator returned invalid {key} evidence.")
    assessment["scores"] = {dimension: round(assessment[dimension], 1) for dimension in dimensions}
    return assessment

def evaluate_tool_usage(expected_tools, tool_calls):
    expected = set(expected_tools)
    attempted = {call["name"] for call in tool_calls if call["status"] in ("completed", "failed")}
    completed = {call["name"] for call in tool_calls if call["status"] == "completed"}
    missing = sorted(expected - completed)
    extra = sorted(attempted - expected)
    failed = sorted({call["name"] for call in tool_calls if call["status"] == "failed"})
    appropriateness = 100 * len(expected & attempted) / len(attempted) if attempted else 0.0
    completeness = 100 * len(expected & completed) / len(expected) if expected else 100.0
    return {
        "scores": {"appropriateness": round(appropriateness, 1), "completeness": round(completeness, 1)},
        "explanations": {
            "appropriateness": f"{len(expected & attempted)} of {len(attempted)} attempted tools were expected; extra tools: {extra}.",
            "completeness": f"{len(expected & completed)} of {len(expected)} expected tools completed; missing or failed tools: {missing}; failed calls: {failed}.",
            "missing_tools": missing,
            "extra_tools": extra,
            "failed_tools": failed,
        },
    }

test_results = []
for scenario in scenarios:
    inputs = {"question": scenario["question"], **scenario.get("inputs", {})}
    result = advisor.invoke(inputs)
    response = result["answer"]
    context = result.get("context", {})
    tool_calls = context.get("tool_calls", [])
    response_assessment = evaluate_response_quality(scenario, response, context)
    tool_assessment = evaluate_tool_usage(scenario["expected_tools"], tool_calls)
    test_results.append({
        "id": scenario["id"],
        "question": scenario["question"],
        "expected_tools": scenario["expected_tools"],
        "expected_response": scenario["expected_response"],
        "decision": {"intent": result.get("intent"), "device": result.get("device"), "target_date": str(result.get("target_date"))},
        "tool_calls": tool_calls,
        "response": response,
        "scores": {**response_assessment["scores"], **tool_assessment["scores"]},
        "explanations": {
            "response": response_assessment["explanations"],
            "response_strengths": response_assessment["strengths"],
            "response_gaps": response_assessment["gaps"],
            "tool_usage": tool_assessment["explanations"],
        },
    })
print(f"Evaluated {len(test_results)} scenarios with LLM response assessments and independent tool-trajectory scores.")

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
def generate_evaluation_report(test_results):
    if not test_results:
        raise ValueError("Cannot generate an evaluation report without scenario results.")
    response_metrics = ("accuracy", "relevance", "completeness", "usefulness")
    tool_metrics = ("appropriateness", "completeness")
    aggregate = {
        "response": {name: round(sum(item["scores"][name] for item in test_results) / len(test_results), 1) for name in response_metrics},
        "tools": {"appropriateness": round(sum(item["scores"]["appropriateness"] for item in test_results) / len(test_results), 1), "completeness": round(sum(item["scores"]["completeness"] for item in test_results) / len(test_results) if test_results else 0, 1)},
    }
    metric_values = [item["scores"][name] for item in test_results for name in response_metrics + tool_metrics]
    pass_rate = 100 * sum(all(item["scores"][name] >= 70 for name in response_metrics) and all(item["scores"][name] >= 80 for name in tool_metrics) for item in test_results) / len(test_results)
    strengths = [f"{item['id']}: " + "; ".join(item["explanations"]["response_strengths"]) for item in test_results if item["explanations"]["response_strengths"]]
    weaknesses = []
    for item in test_results:
        if item["explanations"]["response_gaps"]:
            weaknesses.append(f"{item['id']}: " + "; ".join(item["explanations"]["response_gaps"]))
        if item["scores"]["appropriateness"] < 100 or item["scores"]["completeness"] < 100:
            weaknesses.append(f"{item['id']} tools: " + item["explanations"]["tool_usage"]["appropriateness"] + " " + item["explanations"]["tool_usage"]["completeness"])
    evidence = [{"id": item["id"], "question": item["question"], "decision": item["decision"], "scores": item["scores"], "strengths": item["explanations"]["response_strengths"], "gaps": item["explanations"]["response_gaps"], "tool_assessment": item["explanations"]["tool_usage"], "tool_calls": item["tool_calls"]} for item in test_results]
    recommendations = recommendations_agent.invoke("Generate prioritized improvement recommendations from these actual evaluation results.", {"scenarios": evidence, "aggregate_metrics": aggregate})
    rows = "\n".join("| " + item["id"] + " | " + " | ".join(f"{item['scores'][name]:.1f}%" for name in response_metrics) + f" | {item['scores']['appropriateness']:.1f}% | {item['scores']['completeness']:.1f}% |" for item in test_results)
    metrics_text = "\n".join(["### Response quality", *[f"- {name.title()}: {score:.1f}%" for name, score in aggregate["response"].items()], "### Tool trajectory", *[f"- {name.title()}: {score:.1f}%" for name, score in aggregate["tools"].items()]])
    markdown = f"""# Energy Advisor evaluation report

**Scenarios:** {len(test_results)}  
**Overall mean across response and tool dimensions:** {sum(metric_values) / len(metric_values):.1f}%  
**Pass rate** (each response dimension >= 70%, both tool dimensions >= 80%): {pass_rate:.1f}%

## Overall metrics
{metrics_text}

| Scenario | Accuracy | Relevance | Completeness | Usefulness | Tool appropriateness | Tool completeness |
|---|---:|---:|---:|---:|---:|---:|
{rows}

## Strengths
{chr(10).join('- ' + item for item in strengths) if strengths else '- No response strengths were identified by the evaluator.'}

## Weaknesses and gaps
{chr(10).join('- ' + item for item in weaknesses) if weaknesses else '- No material response or tool-use gaps were identified against these scenarios.'}

## LLM-generated improvement recommendations
{recommendations}
"""
    return {"metrics": aggregate, "pass_rate": pass_rate, "strengths": strengths, "weaknesses": weaknesses, "recommendations": recommendations, "markdown": markdown}

def display_evaluation_report(report):
    display(Markdown(report["markdown"]))

evaluation_report = generate_evaluation_report(test_results)
display_evaluation_report(evaluation_report)